In [ ]:
import numpy as np
from matplotlib import pyplot as plt

### Postlab Q1

The most straightforward way of obtaining a velocity estimate is to use the first-order discrete derivative, also called the Euler backward method, where $\Delta t$ is the sampling period and $x_i$ the position of the ball at timestep i:
$$ v_{x,i} \approx \frac{x_i - x_{i-1}}{\Delta t}$$
The file `pos_logfile.csv` contains a trajectory which was generated by moving the plate by hand and logging the ball coordinates. We will use it to test our velocity signal before we implement it in the C language controller.

Use the provided skeleton to compute the Euler backward estimate of $v_x$ with a sampling frequency of $f = 60Hz$.

Hand in a plot with the velocity estimate in the time range $t \in [10, 15]$ s (Don’t forget the axis labels and plot title). 

Describe your observations, and how they might impact the controller performance.

In [ ]:
f = 60 # Hz
delta_t = 1 / f # s
pos_raw_mm = np.loadtxt("pos_logfile.csv", skiprows=1, delimiter=",")
pos_raw = pos_raw_mm / 1000 # m
v = np.zeros((2463,2)) # m/s

for i in range(len(pos_raw[:,0])):
    if i == 0:
        continue
    v[i, 0] = (pos_raw[i, 0] - pos_raw[i-1, 0] / delta_t)
    v[i, 1] = (pos_raw[i, 1] - pos_raw[i-1, 1] / delta_t)

print(v)

In [ ]:
# Check out how the velocities look like and plot the clipped velocity
t_0 = 10 # s
t_1 = 15 # s
time_values = np.linspace(t_0, t_1, (t_1 - t_0) * 60)

# v_clipped = np.clip(v, -1, 1) # TODO what clip should be used?
plt.figure(figsize=(10, 5))
plt.plot(time_values, v[t_0*60:t_1*60, 0], label="v_x raw")
plt.plot(time_values, v[t_0*60:t_1*60, 1], label="v_y raw")
# plt.plot(v_clipped[t_0*60:t_1*60, 0], label="v_x clipped")
# plt.plot(v_clipped[t_0*60:t_1*60, 1], label="v_y clipped")
plt.title("Clipped Velocity estimate from euler backward ")
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.legend()
plt.grid()
plt.show()

### Postlab Q2

In the IRM lecture, you have learned about the Moving Average (MA) filter. We will use it to improve our velocity estimate. We prevent noise amplification by first filtering the position data and then calculating the velocity instead of filtering the velocity calculated in Q1. For a window size n and unfiltered position values x, the filtered position output y is given by:
$$ y_i = \frac{x_{i-n+1}+ ... + x_i}{n}$$

Use the provided skeleton to compute the filtered velocity signal for $n = [1,5,10,20,30,50,100]$ and compare the resulting signals (window size and delay to unfiltered signal). 

Explain the influence of smaller and larger window sizes on the filter performance and associated delay. Assuming that the delay of the filtered signal should not exceed 0.16s, propose a filter window size which should be implemented in the controller. 

Provide 2 plots with $t \in [10,12]$ s with the following elements:

- Plot 1
    * The unfiltered velocity signal.
    * Comprehensive title and axis labels.
- Plot 2
    * All of the filtered velocity signals.
    * A comprehensive legend, title and axis labels.
    * The legend should include the window size and delay to the unfiltered velocity signal.
- Explanations:
    * Explanation of the influence of the window size on the noise of the signal and associated delay.
    * Proposed window size for the controller.

In [ ]:
# Plot 1: Plot the unfiltered velocity v_x
t_0 = 10 # s
t_1 = 12 # s
time_values = np.linspace(t_0, t_1, (t_1 - t_0) * 60)


plt.figure(figsize=(10, 5))
plt.plot(time_values, v[t_0*60:t_1*60, 0], label="v_x raw")
plt.title("v_x unfiltered estimate form euler backward")
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.legend(loc='lower right')
plt.grid()
plt.show()

In [ ]:
def moving_average(x: np.array, m: int) -> np.array:
    y = np.zeros(x.shape)

    for i in range(len(x)):
        if i < m-1:
            continue
        y[i] = np.mean(x[i-m+1:i+1])

    return y

In [ ]:
# Plot 2: Plot all the filtered velocity signals v_x
n = [1,5,10,20,30,50,100]
pos_x_filtered = np.zeros((len(pos_raw), len(n))) 
v_x_filtered = np.zeros((len(pos_raw), len(n)))

for i, n_val in enumerate(n):
    pos_x_filtered[:, i] = moving_average(pos_raw[:, 0], n_val)

for j in range(len(n)):
    for i in range(len(pos_x_filtered[:, j])):
        if i == 0:
            continue
        v_x_filtered[i, j] = (pos_x_filtered[i, j] - pos_x_filtered[i-1, j] / delta_t)

t_0 = 10 # s
t_1 = 12 # s
time_values = np.linspace(t_0, t_1, (t_1 - t_0) * 60)

plt.figure(figsize=(10, 5))
for i, n_val in enumerate(n):
    plt.plot(time_values, v_x_filtered[t_0*60:t_1*60, i], label="n = {}".format(n_val))
plt.title("v_x filtered estimates form euler backward")
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.legend(loc='lower right')
plt.grid()
plt.show()
